# Lesson 01 - Introduzione agli Agenti AI

Benvenuti alla prima lezione del corso **Agenti AI per Principianti**!

Un **agente AI** è un programma che utilizza un modello di linguaggio ampio (LLM) come motore di ragionamento e può compiere *azioni* nel mondo reale — chiamare API, interrogare database o eseguire codice — per raggiungere un obiettivo per conto di un utente.

In questo notebook costruirai il tuo primo agente: un **Agente di Viaggio** che consiglia destinazioni per le vacanze. Lungo il percorso imparerai come:

1. Connetterti al servizio Azure AI Foundry Agent usando il **Microsoft Agent Framework**.
2. Dare all'agente un **strumento** — una semplice funzione Python che può chiamare.
3. Eseguire l'agente e ispezionare la sua risposta.
4. Trasmettere la risposta dell'agente token per token.


## Setup

Prima di eseguire questo notebook, assicurati di aver:

1. **Un progetto Azure AI Foundry** con un modello chat distribuito (ad esempio `gpt-4o-mini`).
2. **Effettuato l'accesso con Azure CLI** — esegui `az login` nel terminale.
3. **Impostato le variabili d'ambiente richieste:**
   - `AZURE_AI_PROJECT_ENDPOINT` — il tuo endpoint del progetto Azure AI Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — il nome del modello distribuito.

La cella sottostante installa i pacchetti Python necessari.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Creare il tuo primo agente

Un agente ha bisogno di due cose:

- **Istruzioni** che gli dicano *chi è* e *come comportarsi* (un prompt di sistema).
- **Strumenti** — funzioni Python decorate con `@tool` che l’agente può chiamare per recuperare informazioni o eseguire azioni.

Qui sotto definiamo un semplice strumento che restituisce una lista di destinazioni di vacanza popolari. L’agente userà questo strumento quando un utente chiede consigli di viaggio.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Risposte in Streaming

Per un'esperienza più interattiva puoi **streammare** la risposta dell'agente. Invece di aspettare la risposta completa, l'agente fornisce frammenti di testo man mano che vengono generati. Questo è particolarmente utile nelle interfacce di chat dove vuoi mostrare l'output in tempo reale.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

In questa lezione hai imparato a:

- **Creare un provider** che si connette a Azure AI Foundry Agent Service tramite `AzureAIProjectAgentProvider`.
- **Definire uno strumento** usando il decoratore `@tool` in modo che l'agente possa chiamare le tue funzioni Python.
- **Eseguire l'agente** con un messaggio utente e stampare la sua risposta.
- **Trasmettere le risposte** per output in tempo reale.

Nella prossima lezione esploreremo più a fondo i framework agentici e impareremo come fornire agli agenti strumenti più potenti e capacità di ragionamento multi-step.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
